# Variable de confusión, mediadora, colisionador: qué hace ajustar

Complemento de las páginas del curso *Diagramas causales* y *Cómo: dibujar un diagrama causal*.

Simulamos tres pequeños mundos, cada uno con una verdad conocida, y estimamos el efecto de `x` sobre `y` dos veces: **sin** ajustar y **ajustando por** una tercera variable. Qué estimación es la correcta depende por completo del papel de la variable en el diagrama, no de lo bien que se ajuste el modelo.

```
Confounder world:   z -> x,  z -> y,  x -> y     (z is a common cause)
Mediator world:     x -> m -> y,  x -> y         (m is on the path from x to y)
Collider world:     x -> y,  x -> c,  y -> c     (c is a consequence of both)
```

Cambia los parámetros de la siguiente celda y usa *Run All*. Necesita `numpy`, `pandas`, `matplotlib`.

In [ ]:
# ---- Parameters: change these and re-run ----
DIRECT_EFFECT = 2.0     # direct effect of x on y in every world
STRENGTH = 1.0          # how strongly the third variable is tied to x (and to y, in the confounder world)
N = 2000                # sample size
N_SIMS = 300            # simulated datasets per world
SEED = 0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)


def coef_of_x(y, x, *others):
    """Least-squares coefficient on x in y ~ x + others."""
    design = np.column_stack([np.ones_like(x), x, *others])
    return np.linalg.lstsq(design, y, rcond=None)[0][1]


def confounder_world(n):
    z = rng.normal(size=n)
    x = STRENGTH * z + rng.normal(size=n)
    y = DIRECT_EFFECT * x + STRENGTH * z + rng.normal(size=n)
    return y, x, z


def mediator_world(n):
    x = rng.normal(size=n)
    m = STRENGTH * x + rng.normal(size=n)
    y = DIRECT_EFFECT * x + STRENGTH * m + rng.normal(size=n)
    return y, x, m


def collider_world(n):
    x = rng.normal(size=n)
    y = DIRECT_EFFECT * x + rng.normal(size=n)
    c = STRENGTH * x + STRENGTH * y + rng.normal(size=n)
    return y, x, c


# The effect we want in each world is the TOTAL effect of x on y.
worlds = {
    "confounder (z)": (confounder_world, DIRECT_EFFECT),
    "mediator (m)": (mediator_world, DIRECT_EFFECT + STRENGTH * STRENGTH),   # direct + indirect path
    "collider (c)": (collider_world, DIRECT_EFFECT),
}

## 1. ¿Ajustar o no?

Para cada mundo: el efecto total verdadero, la estimación media **sin** ajustar y la estimación media **ajustando por** la tercera variable. La columna correcta debería coincidir con la verdad.

In [ ]:
rows = []
for name, (simulate, truth) in worlds.items():
    unadjusted, adjusted = [], []
    for _ in range(N_SIMS):
        y, x, third = simulate(N)
        unadjusted.append(coef_of_x(y, x))
        adjusted.append(coef_of_x(y, x, third))
    rows.append({"world": name, "true total effect": truth,
                 "unadjusted": np.mean(unadjusted), "adjusting for the third variable": np.mean(adjusted)})
results = pd.DataFrame(rows).set_index("world").round(2)
print(results)

ax = results.plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("estimated effect of x on y")
plt.show()

Cómo leerlo:

- **Variable de confusión:** solo la estimación ajustada recupera la verdad. Si no ajustas, mezclas el efecto de `z` en la estimación.
- **Variable mediadora:** solo la estimación *sin ajustar* da el efecto **total**. Ajustar por `m` elimina la parte del efecto que pasa por ella y deja solo el efecto directo. (Si lo que quieres es el efecto directo, ajustar es lo correcto: la pregunta define la respuesta.)
- **Colisionador:** la estimación sin ajustar es la correcta. Ajustar por `c` crea una asociación que no existe.

## 2. Una variable de confusión que no mediste

En el mundo de la variable de confusión, ¿qué pasa si `z` no está en tu conjunto de datos? Solo podemos hacer el análisis sin ajustar. ¿Ayuda recoger más datos?

In [ ]:
rows = []
for n in [50, 500, 5000, 50000]:
    estimates = [coef_of_x(*confounder_world(n)[:2]) for _ in range(100)]
    rows.append({"n": n, "mean estimate": np.mean(estimates), "SD of estimate": np.std(estimates)})
print(f"True effect: {DIRECT_EFFECT}")
print(pd.DataFrame(rows).set_index("n").round(3))

Más datos hacen la estimación más precisa (la desviación típica se reduce), pero se estabiliza en el valor *equivocado*. Una muestra mayor no puede arreglar una variable de confusión que falta, y por eso el diagrama debe incluir las variables que no mediste y por eso hay que enunciar el supuesto.

## Cosas para probar

- Pon `STRENGTH = 0`: los tres mundos coinciden, porque la tercera variable deja de importar. El sesgo crece con la fuerza de las conexiones.
- Pon `STRENGTH = 2`: la estimación sin ajustar del mundo de la variable de confusión pasa de 2.5 a 2.8, y ajustar por el colisionador da ahora un efecto negativo (alrededor de -0.4) aunque el efecto verdadero es 2.0.
- Pon `N = 100` y `N_SIMS = 1000`: las estimaciones medias se mantienen en los mismos valores, pero los conjuntos de datos individuales se dispersan alrededor de ellos.
- Pon `DIRECT_EFFECT = 0` (sin efecto de `x` sobre `y`): la estimación sin ajustar del mundo del colisionador es 0, mientras que ajustar por `c` produce una asociación negativa espuria. La estimación sin ajustar del mundo de la variable de confusión es distinta de cero por la misma razón: una asociación, no un efecto.